# Stage3 / 00 — Assemble the integrated config

Reads `stage2/metrics/branch_comparison.csv` (produced by stage2/06),
picks the winner, and materializes a concrete stage3 YAML from
`stage3/configs/integrated_placeholder.yaml` by substituting the
`[TODO stage3]` fields with the winner's actual values.

Output: `stage3/configs/integrated_from_<winner>.yaml`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, csv, shutil, re
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)

CSV_IN = os.path.join(REPO_ROOT, 'stage2', 'metrics', 'branch_comparison.csv')
if not os.path.exists(CSV_IN):
    raise FileNotFoundError(f'Run stage2/06 first to produce {CSV_IN}')

# Rank by lane IoU first, mAP@0.5 second — adjust to your deployment goal.
with open(CSV_IN) as f:
    rows = list(csv.DictReader(f))

def key_fn(r):
    try:
        return (float(r.get('lane_IoU', 0) or 0), float(r.get('mAP50', 0) or 0))
    except Exception:
        return (-1, -1)

winner = max(rows, key=key_fn)
print('Stage2 winner:', winner.get('run'), 'lane_IoU=', winner.get('lane_IoU'),
      'mAP50=', winner.get('mAP50'))

# Copy placeholder, patch WARM_START_CKPT path.
src = os.path.join(REPO_ROOT, 'stage3', 'configs', 'integrated_placeholder.yaml')
dst = os.path.join(REPO_ROOT, 'stage3', 'configs', f'integrated_from_{winner["run"]}.yaml')
with open(src) as f: text = f.read()
text = text.replace('<WINNING_BRANCH>', winner['run'])
with open(dst, 'w') as f: f.write(text)
print('Wrote stage3 yaml ->', dst)
print('[TODO] manually resolve remaining `[TODO stage3]` fields based on stage2 run details')